# 🗣️ Multi-Turn Chat with Claude Sonnet 4.6

Welcome to **TechNova Inc.'s AI Help Desk lab**. In this scenario, TechNova wants an assistant that can talk employees through common IT issues—like flaky Wi-Fi, unstable VPN connections, and email problems—without losing the thread of the conversation.

By the end of this notebook, you will be able to:

- Connect Claude Sonnet 4.6 to **Microsoft Foundry**
- Use **system prompts** to define an IT help desk persona
- Build **multi-turn conversations** with shared session memory
- Compare answers **with context vs. without context**
- Control **tone** and **creativity** for different support scenarios


## 🔧 Setting Up Your Connection to Microsoft Foundry

Think of **Microsoft Foundry as a restaurant kitchen**:

- **Claude Sonnet 4.6** is the chef doing the work
- Your **deployment** is that chef's station, with its own setup and capacity
- Your **API key** is your reservation that lets you place an order with the kitchen

In this notebook, we load three environment variables from `../.env`:

- `FOUNDRY_ENDPOINT` — the kitchen door you send requests to
- `FOUNDRY_API_KEY` — the credential that proves you're allowed to order
- `FOUNDRY_MODEL_DEPLOYMENT` — the specific Claude deployment you want to use


In [1]:
import os

from dotenv import load_dotenv
from agent_framework import Agent
from agent_framework.foundry import AnthropicFoundryClient

# Load the Foundry connection details from the repo's .env file.
load_dotenv(dotenv_path="../.env")

# Create a client that talks to Claude Sonnet 4.6 through Microsoft Foundry.
chat_client = AnthropicFoundryClient(
    model=os.environ["FOUNDRY_MODEL_DEPLOYMENT"],
    api_key=os.environ["FOUNDRY_API_KEY"],
    base_url=os.environ["FOUNDRY_ENDPOINT"],
)

# Start with a tiny connectivity check before building the full help desk agent.
connection_test_agent = Agent(
    client=chat_client,
    name="connection-test",
    instructions="You are a concise assistant. Reply in one short sentence.",
)

response = await connection_test_agent.run("Hello, are you there?")
print(response.text)


<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


Yes, I'm here! How can I help you?


## 📋 System Prompts — The Employee Handbook

A **system prompt** is like the employee handbook you give a new hire on day one. It explains:

- who they are
- how they should speak
- what good service looks like
- when they should hand the issue to a human teammate

This matters because Claude is powerful, but the system prompt keeps that power pointed in the right direction. Instead of reinventing its role on every turn, the model consistently behaves like **Nova, TechNova's IT assistant**.


In [2]:
helpdesk_instructions = """
You are Nova, TechNova Inc.'s AI IT support assistant.

Your job is to help employees troubleshoot common IT issues in a professional but friendly tone.
Always:
- ask clear follow-up questions when details are missing
- troubleshoot step by step instead of overwhelming the user
- explain technical ideas in simple language
- summarize the likely cause when you can

Escalate to a human IT specialist when:
- the issue involves account lockouts, security concerns, or sensitive data
- hardware may be physically damaged
- remote troubleshooting does not solve the issue after a few focused steps

Keep responses practical, calm, and easy to follow.
"""

nova_agent = Agent(
    client=chat_client,
    name="nova-helpdesk",
    instructions=helpdesk_instructions,
)

response = await nova_agent.run("My laptop won't connect to the office Wi-Fi.")
print(response.text)


Hi there! I'm Nova, and I'm happy to help you get connected. 👋

Let's figure this out together. Before we start troubleshooting, I have a couple of quick questions:

1. **Is this a new issue**, or did it work before and suddenly stop?
2. **What do you see on your screen** — does your laptop detect the office Wi-Fi network at all, or does it not show up in the list?

This will help me point you in the right direction! 😊


## 🔄 Multi-Turn Conversations — The Notebook That Follows You

Imagine visiting a doctor. Without a medical record, every visit starts from scratch: *What's your name? What changed? Did you already try anything?*

**Sessions are that medical record.** They let Claude remember what happened earlier so the conversation feels natural instead of repetitive.

In the Agent Framework:

- `create_session()` starts a conversation record
- passing `session=session` on each `run()` call keeps building that shared context


In [3]:
session = nova_agent.create_session()

turn_1 = "My VPN keeps disconnecting every 10 minutes when I'm working from home."
response_1 = await nova_agent.run(turn_1, session=session)
print(f"Employee: {turn_1}\n")
print(f"Nova:\n{response_1.text}\n")

turn_2 = "I'm on Windows 11, using GlobalProtect VPN. It started after last week's update."
response_2 = await nova_agent.run(turn_2, session=session)
print(f"Employee: {turn_2}\n")
print(f"Nova:\n{response_2.text}\n")

turn_3 = "Yes, I tried restarting. The error message says 'Gateway timeout'."
response_3 = await nova_agent.run(turn_3, session=session)
print(f"Employee: {turn_3}\n")
print(f"Nova:\n{response_3.text}")


Employee: My VPN keeps disconnecting every 10 minutes when I'm working from home.

Nova:
# VPN Disconnecting Every 10 Minutes

That's frustrating, especially when you're trying to stay productive! The good news is this is a pretty common issue and usually fixable. Let's figure out what's going on.

---

**First, a couple of quick questions to point us in the right direction:**

1. **What device are you using?** (Windows laptop, Mac, company-issued, personal, etc.)
2. **Which VPN client do you use?** (e.g., Cisco AnyConnect, GlobalProtect, Pulse Secure, etc.)
3. **Is your internet connection itself stable?** For example, can you stream a video or browse normally *without* the VPN connected?

---

**While you answer those, here are the most common culprits:**

| Cause | What it means in plain English |
|---|---|
| 🌐 **Unstable home internet** | If your connection dips, the VPN drops |
| ⏱️ **VPN timeout setting** | The VPN is set to disconnect after inactivity |
| 💤 **PC sleep/power sett

## 🚫 What Happens Without Context?

Now let's remove the session.

This is like calling the help desk and getting a different agent each time. Each new person hears only the latest sentence, not the story behind it. That usually means more generic answers, more repeated questions, and less efficient troubleshooting.


In [4]:
no_context_turn_2 = "I'm on Windows 11, using GlobalProtect VPN. It started after last week's update."
no_context_response_2 = await nova_agent.run(no_context_turn_2)
print(f"Employee: {no_context_turn_2}\n")
print(f"Nova without session:\n{no_context_response_2.text}\n")

no_context_turn_3 = "Yes, I tried restarting. The error message says 'Gateway timeout'."
no_context_response_3 = await nova_agent.run(no_context_turn_3)
print(f"Employee: {no_context_turn_3}\n")
print(f"Nova without session:\n{no_context_response_3.text}")


Employee: I'm on Windows 11, using GlobalProtect VPN. It started after last week's update.

Nova without session:
Thanks for those details — that's really helpful! It sounds like the Windows 11 update last week may have interfered with GlobalProtect VPN. This is actually a pretty common situation after system updates.

Before we dive in, can you tell me **what exactly is happening?** For example:

- Does GlobalProtect fail to connect, and if so, is there an error message?
- Does it connect but then drop frequently?
- Does it connect but you can't access company resources?

This will help me point you in the right direction quickly. 😊

Employee: Yes, I tried restarting. The error message says 'Gateway timeout'.

Nova without session:
Thanks for that information! A **"Gateway Timeout"** error (also known as a 504 error) usually means your device is having trouble communicating with a server — it's often a network-related issue rather than something wrong with your computer itself.

Let's

## 🎭 Tone Control — Same Agent, Different Personality

Changing the system prompt changes the **voice**, not the underlying model.

It's like the difference between a casual Slack message and a formal email. Same person, same knowledge, different delivery. This is useful when one audience wants polished enterprise language and another wants something warm and conversational.


In [5]:
formal_agent = Agent(
    client=chat_client,
    name="formal-helpdesk",
    instructions="""
    You are TechNova's enterprise IT support assistant.
    Respond with polished, professional language.
    Use numbered steps and keep the tone formal and reassuring.
    """,
)

casual_agent = Agent(
    client=chat_client,
    name="casual-helpdesk",
    instructions="""
    You are TechNova's friendly IT teammate.
    Sound warm, approachable, and encouraging.
    Keep the advice practical and easy to scan.
    """,
)

issue = "My Outlook email is syncing slowly and attachments won't send."

formal_response = await formal_agent.run(issue)
casual_response = await casual_agent.run(issue)

print("FORMAL VERSION\n")
print(formal_response.text)
print("\n" + "=" * 80 + "\n")
print("CASUAL VERSION\n")
print(casual_response.text)


FORMAL VERSION

# Outlook Synchronization & Attachment Issue — Troubleshooting Guide

Thank you for reaching out to TechNova IT Support. I understand how disruptive email performance issues can be, and I want to assure you that we will resolve this promptly. Please follow the steps below in sequence.

---

## **Part 1: Address Slow Synchronization**

1. **Check your internet connection** — Verify that your network connection is stable and performing at expected speeds.
2. **Restart Outlook** — Close the application completely and relaunch it to refresh the sync process.
3. **Switch to Online Mode** — Navigate to **Send/Receive → Work Offline** and ensure this option is *unchecked*.
4. **Repair your Outlook profile** — Go to **Control Panel → Mail → Show Profiles → Properties → Email Accounts**, then select your account and click **Repair**.
5. **Compact the OST/PST file** — A large data file can slow synchronization. Go to **File → Account Settings → Data Files**, select your file, and

## 🌡️ Temperature — The Creativity Dial

Think of **temperature** as a dial between **"by the book"** and **"creative jazz."**

- **0.0** = more predictable and repeatable
- **1.0** = more variety and experimentation

For IT troubleshooting, lower temperature is usually safer because you want steady, dependable guidance. But for brainstorming user education, communication ideas, or multiple workaround paths, a little more creativity can be helpful.


In [6]:
from agent_framework.anthropic import AnthropicChatOptions

temperature_prompt = "Give me three ways to help an employee whose webcam randomly stops working during video calls."

low_temp_agent = Agent(
    client=chat_client,
    name="low-temp-helpdesk",
    instructions="You are an IT support assistant. Be concise and practical.",
    default_options=AnthropicChatOptions(temperature=0.0, max_tokens=350),
)

high_temp_agent = Agent(
    client=chat_client,
    name="high-temp-helpdesk",
    instructions="You are an IT support assistant. Be concise and practical.",
    default_options=AnthropicChatOptions(temperature=1.0, max_tokens=350),
)

low_run_1 = await low_temp_agent.run(temperature_prompt)
low_run_2 = await low_temp_agent.run(temperature_prompt)
high_run_1 = await high_temp_agent.run(temperature_prompt)
high_run_2 = await high_temp_agent.run(temperature_prompt)

print("LOW TEMPERATURE (0.0) — RUN 1\n")
print(low_run_1.text)
print("\n" + "-" * 80 + "\n")
print("LOW TEMPERATURE (0.0) — RUN 2\n")
print(low_run_2.text)
print("\n" + "=" * 80 + "\n")
print("HIGH TEMPERATURE (1.0) — RUN 1\n")
print(high_run_1.text)
print("\n" + "-" * 80 + "\n")
print("HIGH TEMPERATURE (1.0) — RUN 2\n")
print(high_run_2.text)


LOW TEMPERATURE (0.0) — RUN 1

Here are three practical fixes:

1. **Update/Reinstall the Driver**
Go to Device Manager → find the webcam → right-click and update or uninstall/reinstall the driver. Outdated or corrupt drivers are a common cause.

2. **Check USB Power Management**
Go to Device Manager → USB Root Hub → Properties → Power Management → **uncheck** "Allow the computer to turn off this device to save power." This prevents the system from cutting power to the webcam mid-call.

3. **Test and Switch USB Ports/Bandwidth**
Unplug and try a different USB port (preferably directly on the PC, not a hub). Too many devices sharing a hub can cause dropouts. Also close background apps consuming bandwidth or CPU during calls.

**Quick bonus check:** Ensure the video conferencing app has persistent camera permissions in Windows Privacy Settings.

--------------------------------------------------------------------------------

LOW TEMPERATURE (0.0) — RUN 2

Here are three practical fixes:

## 🎯 Try It Yourself

Try extending TechNova's help desk agent with one or more of these exercises:

1. Add a stricter **escalation policy** for security incidents or repeated failures.
2. Continue the VPN example for **5+ turns** and watch how the session preserves context.
3. Try **low temperature** for precise troubleshooting and **higher temperature** for brainstorming employee self-service guides.
4. Adjust Nova's system prompt so it always ends with a short **next action** summary.

### Key Takeaways

- **Foundry** gives you the hosted connection to Claude deployments.
- **System prompts** define the agent's role, tone, and boundaries.
- **Sessions** are what make multi-turn chat feel coherent instead of forgetful.
- **Tone** can change dramatically even when the model stays the same.
- **Temperature** helps you choose between consistency and creativity.
